# Notebook 04: Exploratory Data Analysis & Visualization

**Mục tiêu:** Khám phá dữ liệu nhiệt độ bề mặt Trái Đất, phát hiện các xu hướng, mẫu mùa vụ, và mối liên hệ giữa các biến để xây dựng nền tảng cho Feature Engineering.

**Quy trình:** Notebook 03 (Data Cleaning) → **Notebook 04 (EDA & Visualization)** → Notebook 05 (Feature Engineering)

---

## I. Giới Thiệu

### Mục tiêu của EDA

Exploratory Data Analysis (EDA) giúp:
1. **Hiểu sâu dữ liệu:** Phân bố, xu hướng, mối quan hệ giữa các biến.
2. **Phát hiện các yếu tố ảnh hưởng:** Xác định các cột có tác động mạnh đến biến mục tiêu hoặc các xu hướng chính.
3. **Chuẩn bị cho Feature Engineering:** Những phát hiện này sẽ được sử dụng để tạo các đặc trưng mới.
4. **Nhận diện vấn đề tiềm ẩn:** Outliers, missing values không được xử lý, hoặc các bất thường cần lưu ý.

### 4 Câu hỏi cốt lõi mà Notebook 04 trả lời

1. **Dữ liệu có đặc điểm gì?** (phân bố, xu hướng, mối quan hệ)
2. **Những yếu tố nào ảnh hưởng đến nhiệt độ?** (thông qua trực quan hóa và phân tích)
3. **Kết quả có ý nghĩa gì đối với bài toán dự báo?** (không chỉ mô tả biểu đồ mà còn giải thích ý nghĩa)
4. **Những phát hiện này sẽ được sử dụng như thế nào ở bước tiếp theo?** (nền tảng cho Feature Engineering)

---

## II. Đọc Dữ Liệu

### 2.1 Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Cấu hình hiển thị
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✓ Thư viện đã được import")

### 2.2 Xác định đường dẫn dữ liệu

In [ ]:
# Tìm project root dựa trên thư mục 'data' và 'notebooks'
def find_project_root():
    current = Path.cwd()
    while current != current.parent:
        # Tìm folder chứa cả 'data' và 'notebooks' hoặc 'notebooks_v1'
        if (current / 'data').exists() and ((current / 'notebooks').exists() or (current / 'notebooks_v1').exists()):
            return current
        current = current.parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
REPORT_IMAGES = PROJECT_ROOT / 'reports' / 'images'

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Processed: {DATA_PROCESSED}")
print(f"Report Images: {REPORT_IMAGES}")

### 2.3 Đọc dữ liệu từ PostgreSQL hoặc CSV

In [ ]:
from dotenv import load_dotenv
import os
import time

DATA_SOURCE = 'UNKNOWN'
df = None

# Tải .env từ các vị trí khác nhau (ưu tiên từ trên xuống)
env_path = None
possible_paths = [
    PROJECT_ROOT / '.env',
    Path.cwd().parent / '.env',
    Path.cwd() / '.env',
]

for path in possible_paths:
    if path.exists():
        env_path = path
        print(f"✓ Tìm thấy .env tại: {path}")
        load_dotenv(env_path)
        break

# Lấy thông tin kết nối
DB_HOST = os.getenv('DB_HOST', '').strip()
DB_PORT = os.getenv('DB_PORT', '').strip()
DB_NAME = os.getenv('DB_NAME', '').strip()
DB_USER = os.getenv('DB_USER', '').strip()
DB_PASSWORD = os.getenv('DB_PASSWORD', '').strip()

# Hiển thị thông tin kết nối
print("\n" + "="*70)
print("🔍 KIỂM TRA KẾT NỐI DATABASE")
print("="*70)
print("\n📋 Thông tin cấu hình:")
print(f"  DB_HOST:     {DB_HOST if DB_HOST else '❌ Chưa cấu hình'}")
print(f"  DB_PORT:     {DB_PORT if DB_PORT else '❌ Chưa cấu hình'}")
print(f"  DB_NAME:     {DB_NAME if DB_NAME else '❌ Chưa cấu hình'}")
print(f"  DB_USER:     {DB_USER if DB_USER else '❌ Chưa cấu hình'}")
print(f"  DB_PASSWORD: {'✓ Được cấu hình' if DB_PASSWORD else '❌ Chưa cấu hình'}")

# Cố kết nối PostgreSQL nếu có đầy đủ thông tin
if all([DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD]):
    try:
        import sqlalchemy
        from sqlalchemy.pool import NullPool
        
        print(f"\n🔌 Đang kết nối PostgreSQL...")
        print(f"   postgresql://{DB_USER}:***@{DB_HOST}:{DB_PORT}/{DB_NAME}")
        start_time = time.time()
        
        connection_string = f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
        
        engine = sqlalchemy.create_engine(
            connection_string,
            connect_args={'connect_timeout': 5},
            poolclass=NullPool,
            echo=False
        )
        
        # Kiểm tra kết nối
        with engine.connect() as conn:
            result = conn.execute(sqlalchemy.text('SELECT 1'))
        
        elapsed = time.time() - start_time
        
        # Đọc dữ liệu từ PostgreSQL
        print(f"   ✓ Kết nối thành công! ({elapsed:.2f}s)")
        print(f"\n📥 Đang tải dữ liệu từ PostgreSQL...")
        
        df = pd.read_sql('SELECT * FROM cleaned_city_temperature', engine)
        
        DATA_SOURCE = 'PostgreSQL ✓'
        print(f"   ✓ Tải dữ liệu thành công!")
        print(f"   📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
        
        engine.dispose()
        
    except TimeoutError:
        print(f"\n   ❌ Timeout (mất quá lâu để kết nối)")
        print(f"   ⏰ Vượt quá 5 giây")
        DATA_SOURCE = 'Timeout → CSV'
        
    except sqlalchemy.exc.OperationalError as e:
        print(f"\n   ❌ Lỗi kết nối ({type(e).__name__})")
        if 'could not connect' in str(e).lower():
            print(f"   📍 Server không phản hồi hoặc không tồn tại")
        elif 'authentication failed' in str(e).lower():
            print(f"   🔐 Sai mật khẩu hoặc user")
        elif 'database' in str(e).lower():
            print(f"   🗄️  Database không tồn tại")
        else:
            print(f"   Lý do: {str(e)[:100]}")
        DATA_SOURCE = 'Error → CSV'
        
    except Exception as e:
        print(f"\n   ❌ Lỗi không xác định ({type(e).__name__})")
        print(f"   Chi tiết: {str(e)[:100]}")
        DATA_SOURCE = 'Error → CSV'

else:
    missing = []
    if not DB_HOST: missing.append('DB_HOST')
    if not DB_PORT: missing.append('DB_PORT')
    if not DB_NAME: missing.append('DB_NAME')
    if not DB_USER: missing.append('DB_USER')
    if not DB_PASSWORD: missing.append('DB_PASSWORD')
    
    print(f"\n⚠️  Cấu hình không đầy đủ:")
    for var in missing:
        print(f"   ❌ {var}")
    DATA_SOURCE = 'Incomplete → CSV'

# Fallback: đọc từ CSV
print(f"\n{'─'*70}")
if df is None:
    print(f"📂 Fallback: Đang tải dữ liệu từ CSV...")
    
    csv_path = DATA_PROCESSED / 'cleaned_city_temperature.csv'
    
    if csv_path.exists():
        start_time = time.time()
        df = pd.read_csv(csv_path)
        elapsed = time.time() - start_time
        
        print(f"   ✓ File tìm thấy: {csv_path}")
        print(f"   ✓ Tải dữ liệu thành công!")
        print(f"   📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
        print(f"   ⏱️  Thời gian tải: {elapsed:.2f}s")
        
        DATA_SOURCE = 'CSV File ✓'
    else:
        print(f"   ❌ File không tìm thấy: {csv_path}")
        print(f"\n   📁 Các file có sẵn trong {DATA_PROCESSED}:")
        
        if DATA_PROCESSED.exists():
            files = list(DATA_PROCESSED.glob('*.csv'))
            if files:
                for f in files[:5]:
                    print(f"      - {f.name}")
                if len(files) > 5:
                    print(f"      ... và {len(files)-5} file khác")
            else:
                print(f"      ⚠️  Không có file CSV nào")
        else:
            print(f"      ⚠️  Folder không tồn tại: {DATA_PROCESSED}")
        
        DATA_SOURCE = 'ERROR: No Data'

print(f"\n{'='*70}")
print(f"📊 NGUỒN DỮ LIỆU: {DATA_SOURCE}")
print(f"{'='*70}\n")

### 2.4 Kiểm tra dữ liệu cơ bản

In [ ]:
print("="*70)
print("📋 KIỂM TRA DỮ LIỆU CƠ BẢN")
print("="*70)

if df is not None and not df.empty:
    print(f"\n✓ Dữ liệu đã được tải thành công!")
    print(f"\n📌 Kích thước dữ liệu:")
    print(f"   Số dòng:  {df.shape[0]:>15,}")
    print(f"   Số cột:   {df.shape[1]:>15}")
    
    print(f"\n📋 Tên các cột ({df.shape[1]} cột):")
    for i, col in enumerate(df.columns, 1):
        dtype = str(df[col].dtype)
        null_count = df[col].isnull().sum()
        null_pct = (null_count / len(df) * 100) if len(df) > 0 else 0
        print(f"   {i:2}. {col:30} ({dtype:10}) - Null: {null_count:8,} ({null_pct:5.1f}%)")
    
    print(f"\n📊 Kiểu dữ liệu:")
    print(df.dtypes)
    
    print(f"\n📈 Mẫu dữ liệu (5 dòng đầu):")
    display(df.head())
    
    print(f"\n📉 Thống kê cơ bản (số liệu):")
    display(df.describe())
    
    print(f"\n✓ Dữ liệu sẵn sàng cho EDA!")
    
elif df is not None:
    print(f"\n⚠️  Dữ liệu trống (0 rows)")
    print(f"   Shape: {df.shape}")
    
else:
    print(f"\n❌ KHÔNG CÓ DỮ LIỆU!")
    print(f"\n📍 Nguyên nhân:")
    print(f"   - PostgreSQL không khả dụng")
    print(f"   - File CSV không tìm thấy: {DATA_PROCESSED / 'cleaned_city_temperature.csv'}")
    print(f"\n💡 Giải pháp:")
    print(f"   1. Kiểm tra PostgreSQL server có chạy không")
    print(f"   2. Hoặc chạy Notebook 03 để tạo file CSV")
    print(f"   3. Hoặc kiểm tra đường dẫn: {DATA_PROCESSED}")

print(f"\n{'='*70}\n")

---

## III. Khám Phá Dữ Liệu (Exploratory Data Analysis)

### 3.1 Correlation Analysis (Phân tích tương quan)

#### 3.1.1 Ma trận tương quan

In [ ]:
if df is not None:
    # Lọc các cột số
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
else:
    numeric_cols = []

if numeric_cols:
    corr_matrix = df[numeric_cols].corr()

    print(f"📊 Ma trận tương quan giữa các biến số:")
    print(corr_matrix)
else:
    print("⚠ Không có cột số để tính tương quan.")

**Trực quan hóa:** vẽ heatmap từ `corr_matrix` đã tính ở cell trên.

In [ ]:
if numeric_cols:
    n_vars = len(corr_matrix)
    fig_size = max(8, min(16, n_vars * 0.8 + 2))
    plt.figure(figsize=(fig_size, fig_size * 0.95))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, cbar_kws={'label': 'Correlation'})
    plt.title('Correlation Matrix - Biểu đồ Tương Quan', fontsize=14, fontweight='bold')
    plt.tight_layout()

    # Lưu ảnh
    if REPORT_IMAGES.exists():
        plt.savefig(REPORT_IMAGES / '01_correlation_heatmap.png', dpi=300, bbox_inches='tight')
        print("✓ Đã lưu: reports/images/01_correlation_heatmap.png")

    plt.show()

#### 3.1.2 Nhận xét tương quan

**Quan sát chính:**
- Các biến nhiệt độ thường có tương quan cao với nhau (ví dụ: land_average_temperature và land_max_temperature).
- Uncertainty (độ không chắc chắn) có thể không tương quan mạnh với các giá trị nhiệt độ nếu chất lượng đo lường không phụ thuộc vào mức độ nhiệt độ.
- **Ý nghĩa cho Feature Engineering:** Nếu tương quan rất cao (> 0.95), có thể xảy ra multicollinearity, cần cân nhắc loại bỏ một trong các biến.

### 3.2 Target Analysis (Phân tích biến mục tiêu)

#### 3.2.1 Xác định biến mục tiêu

Đối với dữ liệu nhiệt độ, **biến mục tiêu (target)** là:
- `city_average_temperature` (nhiệt độ trung bình của thành phố) - **CHÍNH**
- `average_temperature` (nhiệt độ trung bình) - fallback
- `avg_temperature` - fallback

In [ ]:
if df is not None:
    # Xác định cột mục tiêu - ưu tiên city_average_temperature
    target_col = None
    if 'city_average_temperature' in df.columns:
        target_col = 'city_average_temperature'
    elif 'average_temperature' in df.columns:
        target_col = 'average_temperature'
    elif 'avg_temperature' in df.columns:
        target_col = 'avg_temperature'
    
    if target_col:
        print(f"✓ Biến mục tiêu (Target): {target_col}")
        
        # Loại bỏ NaN
        target_data = df[target_col].dropna()
        
        # Thống kê mô tả
        print(f"\n📊 Thống kê của {target_col}:")
        print(f"   Số lượng: {len(target_data):,}")
        print(f"   Trung bình (Mean): {target_data.mean():.2f}°C")
        print(f"   Trung vị (Median): {target_data.median():.2f}°C")
        print(f"   Độ lệch chuẩn (Std): {target_data.std():.2f}")
        print(f"   Min: {target_data.min():.2f}°C")
        print(f"   Max: {target_data.max():.2f}°C")
        print(f"   Khoảng (Range): {target_data.max() - target_data.min():.2f}°C")
        
        # Xác định số bins phù hợp
        n_data = len(target_data)
        n_bins = max(20, min(100, n_data // 100 + 30))
        
        # Histogram
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Histogram
        axes[0].hist(target_data, bins=n_bins, color='steelblue', edgecolor='black', alpha=0.7)
        axes[0].axvline(target_data.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {target_data.mean():.2f}°C')
        axes[0].axvline(target_data.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {target_data.median():.2f}°C')
        axes[0].set_xlabel(target_col, fontsize=11)
        axes[0].set_ylabel('Tần số (Frequency)', fontsize=11)
        axes[0].set_title(f'Phân bố {target_col} (Histogram)', fontsize=12, fontweight='bold')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Boxplot
        axes[1].boxplot(target_data, vert=True)
        axes[1].set_ylabel(target_col, fontsize=11)
        axes[1].set_title(f'Phân bố {target_col} (Boxplot)', fontsize=12, fontweight='bold')
        axes[1].grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        
        # Lưu ảnh
        if REPORT_IMAGES.exists():
            plt.savefig(REPORT_IMAGES / '02_target_distribution.png', dpi=300, bbox_inches='tight')
            print("\n✓ Đã lưu: reports/images/02_target_distribution.png")
        
        plt.show()
    else:
        print("⚠ Không tìm thấy cột mục tiêu (city_average_temperature, average_temperature hoặc avg_temperature)")

#### 3.2.2 Nhận xét về phân bố biến mục tiêu

**Quan sát:**
- **Hình dạng phân bố:** Bình thường, lệch trái/phải, hoặc nhiều peak?
- **Outliers:** Có nhiệt độ bất thường (quá cao/quá thấp) không?
- **Ý nghĩa:** Phân bố cân bằng là tốt cho mô hình hồi quy. Nếu lệch, có thể cần transform (log, sqrt).
- **Ứng dụng cho ML:** Nếu phân bố bình thường, các mô hình tuyến tính sẽ hiệu quả hơn.

### 3.3 Univariate Analysis (Phân tích từng biến riêng lẻ)

#### Biểu đồ Interactive - Các Cột Danh Mục

Phần này phân tích **từng cột danh mục riêng lẻ**, chia thành 3 nhóm để chọn cách trực quan phù hợp nhất — thay vì dùng chung 1 kiểu bar chart cho mọi cột (không phù hợp với cột ngày tháng hay cột định danh có quá nhiều giá trị):

1. **🗓️ Nhóm thời gian** (`observation_date`): parse sang datetime, xem khoảng thời gian bao phủ + biểu đồ đường số bản ghi theo năm — thay vì bar chart 1.800+ ngày riêng lẻ vô nghĩa.
2. **🏙️ Nhóm định danh nhiều giá trị** (`city_name`, > 100 categories): Top 10 phổ biến nhất + histogram phân bố số bản ghi/giá trị, để thấy đây là identifier chứ không phải nhóm mang ý nghĩa phân loại.
3. **📋 Nhóm danh mục thông thường** (`country_name`...): bar chart Top 15 như trước, vì cardinality thấp nên biểu đồ này vẫn trực quan và hữu ích.

**Cách đánh giá biến (áp dụng cho Nhóm 3):**
- 🟢 **CHÍNH** (≤5): Dùng ngay One-Hot Encoding
- 🟡 **PHỤ** (6-20): Có thể dùng, cần xem xét reduce

In [ ]:
if df is not None:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    all_object_cols = df.select_dtypes(include='object').columns.tolist()

    if all_object_cols:
        # Phân loại 3 nhóm để chọn cách trực quan phù hợp cho từng loại cột
        HIGH_CARD_THRESHOLD = 100
        date_cols = [c for c in all_object_cols if 'date' in c.lower()]
        high_card_cols = [c for c in all_object_cols if c not in date_cols and df[c].nunique() > HIGH_CARD_THRESHOLD]
        normal_cat_cols = [c for c in all_object_cols if c not in date_cols and c not in high_card_cols]

        important_vars = []
        weak_vars = []

        # ========== NHÓM 1: CỘT THỜI GIAN (vd: observation_date) ==========
        if date_cols:
            print("="*80)
            print("🗓️  NHÓM 1: CỘT THỜI GIAN (không phù hợp bar chart theo category)")
            print("="*80)

            for col in date_cols:
                parsed = pd.to_datetime(df[col], errors='coerce')
                n_valid = parsed.notna().sum()

                print(f"\n🔹 CỘT: {col}")
                print(f"   Kiểu dữ liệu gốc: {df[col].dtype} → đã convert sang datetime để phân tích")
                print(f"   Khoảng thời gian: {parsed.min().date()} → {parsed.max().date()}")
                print(f"   Số năm bao phủ: {parsed.dt.year.max() - parsed.dt.year.min() + 1} năm")
                print(f"   Số mốc thời gian duy nhất: {parsed.nunique():,}")
                print(f"   Parse thành công: {n_valid:,}/{len(parsed):,} ({n_valid/len(parsed)*100:.1f}%)")
                print(f"   💡 Đã có sẵn year/month/quarter/decade tách riêng để phân tích xu hướng & mùa vụ (mục 3.6)")
                weak_vars.append(col)

                yearly_counts = parsed.dt.year.value_counts().sort_index()

                fig_date = go.Figure()
                fig_date.add_trace(go.Scatter(
                    x=yearly_counts.index, y=yearly_counts.values,
                    mode='lines+markers', fill='tozeroy',
                    line=dict(color='#2563eb', width=2),
                    marker=dict(size=4),
                    hovertemplate='Năm %{x}<br>Số bản ghi: %{y:,}<extra></extra>'
                ))
                fig_date.update_layout(
                    title=f"📅 Độ phủ dữ liệu theo thời gian — {col}",
                    xaxis_title="Năm", yaxis_title="Số bản ghi",
                    template='plotly_white', height=350
                )
                fig_date.show()
                print(f"   📈 Số bản ghi tăng dần theo năm chủ yếu do số thành phố được ghi nhận tăng dần,")
                print(f"      không phải do biến động khí hậu → cần lưu ý coverage bias khi diễn giải xu hướng dài hạn.")

        # ========== NHÓM 2: CỘT ĐỊNH DANH NHIỀU GIÁ TRỊ (vd: city_name) ==========
        if high_card_cols:
            print("\n" + "="*80)
            print(f"🏙️  NHÓM 2: CỘT ĐỊNH DANH NHIỀU GIÁ TRỊ (> {HIGH_CARD_THRESHOLD} categories)")
            print("="*80)

            for col in high_card_cols:
                value_counts_full = df[col].value_counts()
                n_unique = df[col].nunique()
                top10 = value_counts_full.head(10)

                print(f"\n🔹 CỘT: {col}")
                print(f"   Kiểu dữ liệu: {df[col].dtype}")
                print(f"   Số giá trị duy nhất: {n_unique:,}")
                print(f"   Missing: {df[col].isna().sum()} ({df[col].isna().sum()/len(df)*100:.2f}%)")
                print(f"   Số bản ghi/giá trị: trung bình={value_counts_full.mean():.1f}, "
                      f"min={value_counts_full.min()}, max={value_counts_full.max()}")
                print(f"   Top 5 giá trị phổ biến nhất:")
                for i, (val, count) in enumerate(top10.head(5).items(), 1):
                    pct = (count / len(df)) * 100
                    print(f"      {i}. {val}: {count:,} ({pct:.1f}%)")
                print(f"   💡 Mức độ quan trọng: 🔴 THẤP (quá nhiều categories để one-hot encode trực tiếp)")
                weak_vars.append(col)

                fig_hc = make_subplots(
                    rows=1, cols=2,
                    subplot_titles=[f"Top 10 {col} phổ biến nhất", f"Phân bố số bản ghi mỗi {col}"],
                    column_widths=[0.5, 0.5]
                )
                fig_hc.add_trace(
                    go.Bar(
                        x=top10.values, y=top10.index, orientation='h',
                        marker=dict(color='#1f77b4'),
                        text=[f"{v:,}" for v in top10.values], textposition='auto',
                        hovertemplate='<b>%{y}</b><br>Count: %{x:,}<extra></extra>',
                        showlegend=False
                    ), row=1, col=1
                )
                fig_hc.add_trace(
                    go.Histogram(
                        x=value_counts_full.values, nbinsx=30,
                        marker=dict(color='#f97316'),
                        hovertemplate='Số bản ghi: %{x}<br>Số lượng giá trị: %{y}<extra></extra>',
                        showlegend=False
                    ), row=1, col=2
                )
                fig_hc.update_xaxes(title_text="Count", row=1, col=1)
                fig_hc.update_xaxes(title_text="Số bản ghi", row=1, col=2)
                fig_hc.update_yaxes(title_text="Số lượng giá trị", row=1, col=2)
                fig_hc.update_layout(height=400, template='plotly_white',
                                      title_text=f"📊 {col} — {n_unique:,} giá trị duy nhất")
                fig_hc.show()
                print(f"   📈 Biểu đồ phải: nếu các cột dồn quanh 1 giá trị → hầu hết {col} có số bản ghi gần bằng nhau,")
                print(f"      xác nhận đây là identifier (định danh), không phải nhóm mang ý nghĩa phân loại riêng.")

                # Bổ sung: Top nóng nhất / lạnh nhất theo biến mục tiêu (insight khí hậu thực sự, không chỉ đếm dòng)
                if 'target_col' in globals() and target_col and target_col in df.columns:
                    city_avg_temp = df.groupby(col)[target_col].mean().dropna().sort_values()

                    if len(city_avg_temp) >= 10:
                        coldest10 = city_avg_temp.head(10)
                        hottest10 = city_avg_temp.tail(10).sort_values(ascending=False)

                        print(f"\n   🌡️  TOP 10 {col.upper()} NÓNG NHẤT & LẠNH NHẤT (theo {target_col} trung bình):")
                        print(f"\n   🔥 Nóng nhất:")
                        for i, (city, temp) in enumerate(hottest10.items(), 1):
                            print(f"      {i}. {city}: {temp:.2f}°C")
                        print(f"\n   🧊 Lạnh nhất:")
                        for i, (city, temp) in enumerate(coldest10.items(), 1):
                            print(f"      {i}. {city}: {temp:.2f}°C")

                        top20 = pd.concat([coldest10, hottest10]).sort_values()
                        colors_temp = ['#3b82f6' if city in coldest10.index else '#ef4444' for city in top20.index]

                        fig_temp = go.Figure(go.Bar(
                            x=top20.values, y=top20.index, orientation='h',
                            marker=dict(color=colors_temp),
                            text=[f"{t:.1f}°C" for t in top20.values], textposition='auto',
                            hovertemplate='<b>%{y}</b><br>Nhiệt độ TB: %{x:.2f}°C<extra></extra>'
                        ))
                        fig_temp.update_layout(
                            title=f"🌡️ Top 10 {col} nóng nhất (đỏ) & lạnh nhất (xanh) — theo {target_col} trung bình",
                            xaxis_title=f"{target_col} trung bình (°C)", yaxis_title=col,
                            template='plotly_white', height=500
                        )
                        fig_temp.show()

                        temp_range = hottest10.max() - coldest10.min()
                        print(f"\n   💡 Chênh lệch giữa {col} nóng nhất và lạnh nhất: {temp_range:.2f}°C")
                        print(f"      → Xác nhận vị trí địa lý là yếu tố quyết định mạnh đến nhiệt độ,")
                        print(f"      → nên kết hợp {col} với country_name/tọa độ khi mã hóa vị trí (xem checklist Notebook 05).")

        # ========== NHÓM 3: CỘT DANH MỤC THÔNG THƯỜNG (vd: country_name) ==========
        if normal_cat_cols:
            print("\n" + "="*80)
            print("📋 NHÓM 3: CỘT DANH MỤC THÔNG THƯỜNG (bar chart Top categories)")
            print("="*80)

            n_plots = len(normal_cat_cols)
            fig = make_subplots(
                rows=n_plots, cols=1,
                subplot_titles=[f"Phân bố {col}" for col in normal_cat_cols],
                vertical_spacing=(0.12 / n_plots) if n_plots > 1 else 0.1
            )

            for idx, col in enumerate(normal_cat_cols, 1):
                value_counts = df[col].value_counts().head(15)
                n_unique = df[col].nunique()

                print(f"\n🔹 CỘT: {col}")
                print(f"   Kiểu dữ liệu: {df[col].dtype}")
                print(f"   Số giá trị duy nhất: {n_unique}")
                print(f"   Missing: {df[col].isna().sum()} ({df[col].isna().sum()/len(df)*100:.2f}%)")
                print(f"   Top 5 giá trị:")

                importance_level = "🟢 CHÍNH" if n_unique <= 5 else "🟡 PHỤ"
                important_vars.append((col, n_unique))

                for i, (val, count) in enumerate(value_counts.head(5).items(), 1):
                    pct = (count / len(df)) * 100
                    print(f"      {i}. {val}: {count:,} ({pct:.1f}%)")

                print(f"   💡 Mức độ quan trọng: {importance_level}")

                colors = ['#1f77b4' if i == 0 else '#aec7e8' for i in range(len(value_counts))]
                fig.add_trace(
                    go.Bar(
                        x=value_counts.values, y=value_counts.index, orientation='h',
                        marker=dict(color=colors, line=dict(color='darkblue', width=1)),
                        name=col, text=[f"{v:,}" for v in value_counts.values], textposition='auto',
                        hovertemplate='<b>%{y}</b><br>Count: %{x:,}<extra></extra>', showlegend=False
                    ), row=idx, col=1
                )
                fig.update_xaxes(title_text="Count", row=idx, col=1)
                fig.update_yaxes(title_text=col, row=idx, col=1)

            fig.update_layout(
                height=300 * n_plots, showlegend=False,
                title_text="📊 Phân tích Các Cột Danh Mục Thông Thường (Interactive)",
                hovermode='closest', template='plotly_white'
            )
            fig.show()

            if REPORT_IMAGES.exists():
                html_path = REPORT_IMAGES / '03_categorical_analysis_interactive.html'
                fig.write_html(str(html_path))
                print(f"\n✓ Đã lưu (Interactive): {html_path}")

        # ========== TỔNG KẾT ==========
        print("\n" + "="*80)
        print("📊 TỔNG KẾT TOÀN BỘ CÁC CỘT DANH MỤC")
        print("="*80)

        if important_vars:
            print(f"\n🟢 BIẾN NÊN DÙNG TRỰC TIẾP LÀM FEATURE ({len(important_vars)} cột):\n")
            for col_name, n_unique in important_vars:
                print(f"   ✓ CỘT: '{col_name}' — {n_unique} categories")
                print(f"     → {'One-Hot Encoding' if n_unique <= 5 else 'Frequency/Target Encoding hoặc reduce categories'}")

        if weak_vars:
            print(f"\n🔴 BIẾN CẦN XỬ LÝ RIÊNG, KHÔNG ENCODE TRỰC TIẾP ({len(weak_vars)} cột):\n")
            for col_name in weak_vars:
                reason = "cột thời gian → dùng year/month/quarter thay vì raw string" if col_name in date_cols \
                    else f"{df[col_name].nunique():,} categories → quá nhiều, cần group/target encoding"
                print(f"   ○ CỘT: '{col_name}' — {reason}")

        print("\n" + "="*80)
        print(f"📌 Tổng: {len(all_object_cols)} cột object = {len(date_cols)} thời gian + "
              f"{len(high_card_cols)} định danh nhiều giá trị + {len(normal_cat_cols)} danh mục thường")
        print("="*80)
    else:
        print("⚠️  Không có cột danh mục (object type) trong dữ liệu")
else:
    print("❌ Không có dữ liệu để phân tích (df is None)")

🗓️  NHÓM 1: CỘT THỜI GIAN (không phù hợp bar chart theo category)

🔹 CỘT: observation_date
   Kiểu dữ liệu gốc: object → đã convert sang datetime để phân tích
   Khoảng thời gian: 1863-01-01 → 2013-09-01
   Số năm bao phủ: 151 năm
   Số mốc thời gian duy nhất: 1,809
   Parse thành công: 5,579,085/5,579,085 (100.0%)
   💡 Đã có sẵn year/month/quarter/decade tách riêng để phân tích xu hướng & mùa vụ (mục 3.6)


   📈 Số bản ghi tăng dần theo năm chủ yếu do số thành phố được ghi nhận tăng dần,
      không phải do biến động khí hậu → cần lưu ý coverage bias khi diễn giải xu hướng dài hạn.

🏙️  NHÓM 2: CỘT ĐỊNH DANH NHIỀU GIÁ TRỊ (> 100 categories)

🔹 CỘT: city_name
   Kiểu dữ liệu: str
   Số giá trị duy nhất: 3,070
   Missing: 0 (0.00%)
   Số bản ghi/giá trị: trung bình=1817.3, min=1464, max=5427
   Top 5 giá trị phổ biến nhất:
      1. Springfield: 5,427 (0.1%)
      2. Rongcheng: 5,422 (0.1%)
      3. Worcester: 5,420 (0.1%)
      4. Arlington: 3,618 (0.1%)
      5. Aurora: 3,618 (0.1%)
   💡 Mức độ quan trọng: 🔴 THẤP (quá nhiều categories để one-hot encode trực tiếp)


   📈 Biểu đồ phải: nếu các cột dồn quanh 1 giá trị → hầu hết city_name có số bản ghi gần bằng nhau,
      xác nhận đây là identifier (định danh), không phải nhóm mang ý nghĩa phân loại riêng.

   🌡️  TOP 10 CITY_NAME NÓNG NHẤT & LẠNH NHẤT (theo city_average_temperature trung bình):

   🔥 Nóng nhất:
      1. Tirupati: 28.59°C
      2. Pallavaram: 28.59°C
      3. Madras: 28.59°C
      4. Avadi: 28.59°C
      5. Kanchipuram: 28.59°C
      6. Tiruvottiyur: 28.59°C
      7. Tambaram: 28.59°C
      8. Ambattur: 28.59°C
      9. Alandur: 28.59°C
      10. Ongole: 28.40°C

   🧊 Lạnh nhất:
      1. Norilsk: -11.72°C
      2. Kyzyl: -6.05°C
      3. Chita: -4.18°C
      4. Ust Ilimsk: -3.82°C
      5. Surgut: -3.33°C
      6. Nefteyugansk: -3.33°C
      7. Ulan Ude: -3.00°C
      8. Bratsk: -2.64°C
      9. Yakeshi: -2.33°C
      10. Hailar: -2.33°C



   💡 Chênh lệch giữa city_name nóng nhất và lạnh nhất: 40.31°C
      → Xác nhận vị trí địa lý là yếu tố quyết định mạnh đến nhiệt độ,
      → nên kết hợp city_name với country_name/tọa độ khi mã hóa vị trí (xem checklist Notebook 05).

📋 NHÓM 3: CỘT DANH MỤC THÔNG THƯỜNG (bar chart Top categories)

🔹 CỘT: country_name
   Kiểu dữ liệu: str
   Số giá trị duy nhất: 50
   Missing: 0 (0.00%)
   Top 5 giá trị:
      1. India: 693,711 (12.4%)
      2. China: 686,966 (12.3%)
      3. United States: 464,897 (8.3%)
      4. Brazil: 392,342 (7.0%)
      5. Japan: 316,032 (5.7%)
   💡 Mức độ quan trọng: 🟡 PHỤ



✓ Đã lưu (Interactive): c:\Users\COMPUTER\Desktop\du_an_2\Global-Surface-Temperature-Analysis\reports\images\03_categorical_analysis_interactive.html

📊 TỔNG KẾT TOÀN BỘ CÁC CỘT DANH MỤC

🟢 BIẾN NÊN DÙNG TRỰC TIẾP LÀM FEATURE (1 cột):

   ✓ CỘT: 'country_name' — 50 categories
     → Frequency/Target Encoding hoặc reduce categories

🔴 BIẾN CẦN XỬ LÝ RIÊNG, KHÔNG ENCODE TRỰC TIẾP (2 cột):

   ○ CỘT: 'observation_date' — cột thời gian → dùng year/month/quarter thay vì raw string
   ○ CỘT: 'city_name' — 3,070 categories → quá nhiều, cần group/target encoding

📌 Tổng: 3 cột object = 1 thời gian + 1 định danh nhiều giá trị + 1 danh mục thường


: 

### 3.4 Correlation with Target Variable (Tương quan với biến mục tiêu)

In [ ]:
if df is not None and target_col:
    # Tính tương quan của các biến số với target
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_cols:
        correlations = df[numeric_cols].corr()[target_col].sort_values(ascending=False)
        
        print(f"\n📊 Tương quan của các biến với {target_col}:")
        print(correlations)
        
        # Tính kích thước phù hợp
        n_vars = len(correlations)
        fig_height = max(5, n_vars * 0.3)
        
        # Biểu đồ thanh
        fig, ax = plt.subplots(figsize=(10, fig_height))
        colors = ['green' if x > 0 else 'red' for x in correlations.values]
        ax.barh(range(len(correlations)), correlations.values, color=colors, alpha=0.7)
        ax.set_yticks(range(len(correlations)))
        ax.set_yticklabels(correlations.index, fontsize=10)
        ax.set_xlabel('Correlation Coefficient', fontsize=11)
        ax.set_title(f'Tương quan với {target_col}', fontsize=12, fontweight='bold')
        ax.axvline(0, color='black', linestyle='-', linewidth=0.5)
        ax.grid(True, alpha=0.3, axis='x')
        
        plt.tight_layout()
        
        # Lưu ảnh
        if REPORT_IMAGES.exists():
            plt.savefig(REPORT_IMAGES / '04_correlation_with_target.png', dpi=300, bbox_inches='tight')
            print("✓ Đã lưu: reports/images/04_correlation_with_target.png")
        
        plt.show()
        
        # Nhận xét
        print("\n💡 Nhận xét:")
        strong_corr = correlations[correlations.abs() > 0.7]
        if len(strong_corr) > 0:
            print(f"   Biến có tương quan mạnh (|r| > 0.7):")
            for var, corr in strong_corr.items():
                print(f"   - {var}: {corr:.3f}")
        else:
            print(f"   Không có biến nào có tương quan mạnh (|r| > 0.7) với {target_col}")

### 3.5 Multivariate Analysis (Phân tích tương tác giữa các biến)

#### 3.5.1 Phân tích Scatter Plot - Các Biến vs Biến Mục Tiêu

In [ ]:
if df is not None and target_col:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [col for col in numeric_cols if col != target_col][:6]

    if feature_cols:
        print(f"📊 Tạo {len(feature_cols)} scatter plots tương quan...\n")

        # Tạo grid subplot (2 hàng x 3 cột) - NHANH HƠN so với vẽ từng cái
        n_cols = 3
        n_rows = (len(feature_cols) + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
        axes = axes.flatten()

        correlations_list = []

        for idx, col in enumerate(feature_cols):
            ax = axes[idx]
            valid_data = df[[col, target_col]].dropna()

            if len(valid_data) > 0:
                ax.scatter(valid_data[col], valid_data[target_col], 
                          alpha=0.3, s=15, color='steelblue', edgecolors='none')

                correlation = valid_data[col].corr(valid_data[target_col])
                correlations_list.append((col, correlation, len(valid_data)))

                if len(valid_data) > 10:
                    z = np.polyfit(valid_data[col], valid_data[target_col], 1)
                    p = np.poly1d(z)
                    x_line = np.linspace(valid_data[col].min(), valid_data[col].max(), 50)
                    ax.plot(x_line, p(x_line), "r-", alpha=0.8, linewidth=2)

                title_color = 'darkgreen' if abs(correlation) > 0.5 else 'orange' if abs(correlation) > 0.3 else 'gray'
                ax.set_xlabel(col, fontsize=10, fontweight='bold')
                ax.set_ylabel(target_col, fontsize=10, fontweight='bold')
                ax.set_title(f'{col}\nr={correlation:+.3f} (N={len(valid_data):,})',
                            fontsize=11, fontweight='bold', color=title_color)
                ax.grid(True, alpha=0.2)

        for idx in range(len(feature_cols), len(axes)):
            axes[idx].axis('off')

        plt.suptitle('Phân tích Tương quan Đa Biến (Multivariate Analysis)',
                     fontsize=14, fontweight='bold', y=0.995)
        plt.tight_layout()

        if REPORT_IMAGES.exists():
            plt.savefig(REPORT_IMAGES / '05_multivariate_scatter_grid.png', dpi=300, bbox_inches='tight')
            print("✓ Đã lưu: reports/images/05_multivariate_scatter_grid.png\n")

        plt.show()

        print("\n" + "="*80)
        print("📊 NHẬN XÉT PHÂN TÍCH TƯƠNG QUAN ĐA BIẾN (MULTIVARIATE ANALYSIS)")
        print("="*80)

        correlations_list.sort(key=lambda x: abs(x[1]), reverse=True)

        print("\n🔍 XẾP HẠNG BIẾN THEO ĐỘ MẠNH TƯƠNG QUAN VỚI BIẾN MỤC TIÊU:\n")

        for rank, (col, corr, n) in enumerate(correlations_list, 1):
            strength = "🟢 MẠNH (r>0.7)" if abs(corr) > 0.7 else "🟡 VỪA (0.4<|r|≤0.7)" if abs(corr) > 0.4 else "🔴 YẾU (|r|≤0.4)"
            direction = "↗ tăng" if corr > 0 else "↘ giảm"

            print(f"{rank}. {col.upper()}")
            print(f"   Tương quan: r = {corr:+.4f} {strength} - {direction}")
            print(f"   Dữ liệu: {n:,} mẫu (không NaN)")

            if abs(corr) > 0.7:
                print(f"   ✓ STRONG PREDICTOR - Biến này là một yếu tố dự báo VỀ CHÍNH.")
                print(f"     • Sử dụng ngay trong mô hình Machine Learning")
                print(f"     • Kiểm tra multicollinearity")
            elif abs(corr) > 0.4:
                print(f"   • MODERATE PREDICTOR - Có giá trị nhưng không phải factor chính.")
                print(f"     • Kết hợp với các biến khác hoặc tạo interaction terms")
            else:
                print(f"   ○ WEAK PREDICTOR - Tương quan yếu nhưng vẫn có ý nghĩa.")
                print(f"     • Tạo polynomial features hoặc interaction")

            print()

        print("="*80)
        print("\n💡 NHẬN ĐỊNH TỔNG THỂ:")
        strong_vars = [col for col, corr, n in correlations_list if abs(corr) > 0.5]
        if strong_vars:
            print(f"✓ Có {len(strong_vars)} biến mạnh: {', '.join(strong_vars)}")
            print(f"  → Ưu tiên trong mô hình Machine Learning")
        else:
            print(f"⚠ KHÔNG CÓ biến mạnh (r>0.5). Cần feature engineering phức tạp.")
        print("="*80)

In [ ]:
if df is not None and target_col and len(feature_cols) >= 2:
    print("🔗 Phân tích Feature Interaction (Tương tác giữa các biến)...\n")

    interactions = []
    
    for i in range(len(feature_cols)):
        for j in range(i+1, min(i+3, len(feature_cols))):
            col1, col2 = feature_cols[i], feature_cols[j]
            
            valid_idx = df[[col1, col2, target_col]].notna().all(axis=1)
            if valid_idx.sum() > 0:
                df_temp = df[valid_idx].copy()
                df_temp['interaction'] = df_temp[col1] * df_temp[col2]
                
                corr_interaction = df_temp['interaction'].corr(df_temp[target_col])
                corr1 = df_temp[col1].corr(df_temp[target_col])
                corr2 = df_temp[col2].corr(df_temp[target_col])
                max_individual = max(abs(corr1), abs(corr2))
                
                interactions.append({
                    'pair': f"{col1} × {col2}",
                    'corr_interaction': corr_interaction,
                    'corr_individual_max': max_individual,
                    'improvement': abs(corr_interaction) - max_individual
                })
    
    if interactions:
        interactions.sort(key=lambda x: abs(x['improvement']), reverse=True)
        
        print("="*80)
        print("🔍 TOP FEATURE INTERACTIONS - TƯƠNG TỰ GIÚP TĂNG DỰ BÁO")
        print("="*80)
        print("\nCách đọc: Nếu improvement > 0, kết hợp 2 biến sẽ tạo feature mạnh hơn.\n")
        
        for idx, inter in enumerate(interactions[:5], 1):
            print(f"{idx}. {inter['pair']}")
            print(f"   r(interaction) = {inter['corr_interaction']:+.4f}")
            print(f"   r(max individual) = {inter['corr_individual_max']:+.4f}")
            print(f"   ΔΔ improvement = {inter['improvement']:+.4f}")
            
            if inter['improvement'] > 0.05:
                print(f"   ✓ IMPROVEMENT - Tương tác này TỐT HƠN!")
                print(f"     → Tạo feature này ở Notebook 05")
            elif inter['improvement'] > 0:
                print(f"   ~ Slight improvement - Có thể tạo")
            else:
                print(f"   ✗ No improvement")
            print()
        
        print("="*80)

### 3.6 Time Series Analysis (Phân tích theo thời gian)

In [ ]:
if df is not None and target_col and 'year' in df.columns:
    # Tính trung bình theo năm
    yearly_avg = df.groupby('year')[target_col].agg(['mean', 'std', 'count']).reset_index()
    
    print(f"\n📈 Xu hướng {target_col} theo năm:")
    print(yearly_avg.head(10))
    
    # Tính kích thước phù hợp
    n_years = len(yearly_avg)
    fig_width = max(12, n_years * 0.08 + 2)
    
    # Biểu đồ đường
    fig, ax = plt.subplots(figsize=(fig_width, 6))
    ax.plot(yearly_avg['year'], yearly_avg['mean'], marker='o', linewidth=2, markersize=4, label='Mean')
    ax.fill_between(yearly_avg['year'], 
                     yearly_avg['mean'] - yearly_avg['std'], 
                     yearly_avg['mean'] + yearly_avg['std'], 
                     alpha=0.2, label='±1 Std Dev')
    ax.set_xlabel('Năm (Year)', fontsize=11)
    ax.set_ylabel(f'{target_col} (°C)', fontsize=11)
    ax.set_title(f'Xu hướng {target_col} theo Năm', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Lưu ảnh
    if REPORT_IMAGES.exists():
        plt.savefig(REPORT_IMAGES / '06_yearly_trend.png', dpi=300, bbox_inches='tight')
        print("\n✓ Đã lưu: reports/images/06_yearly_trend.png")
    
    plt.show()
    
    # Tính xu hướng
    trend_slope = (yearly_avg['mean'].iloc[-1] - yearly_avg['mean'].iloc[0]) / len(yearly_avg)
    print(f"\n📊 Xu hướng: {trend_slope:.4f}°C/năm")
    if trend_slope > 0:
        print(f"   ⬆️  Nhiệt độ tăng theo thời gian (nóng lên)")
    else:
        print(f"   ⬇️  Nhiệt độ giảm theo thời gian (lạnh lên)")

### 4.1 Các Phát Hiện Chính

Sau khi phân tích **5.579.085 dòng** dữ liệu (32 cột, giai đoạn 1863–2013, 50 quốc gia, 3.119 vị trí thành phố), những phát hiện chính:

**1. Phân bố biến mục tiêu (`city_average_temperature`):**
- Mean = 17.50°C, Median = 19.83°C, Std = 10.19°C, Min = -42.70°C, Max = 39.16°C (khoảng biến động 81.86°C).
- Mean < Median → phân bố **lệch trái (left-skewed)**: phần lớn quan sát tập trung ở vùng nhiệt độ ấm, nhưng có một đuôi dài kéo về phía nhiệt độ rất thấp (mùa đông, vùng vĩ độ cao) làm giá trị trung bình bị kéo xuống dưới trung vị. Chi tiết xem `reports/images/02_target_distribution.png`.
- Các giá trị cực trị này không bị cắt/xóa vì Notebook 03 đã xác nhận đây là outlier khí hậu hợp lệ (đã gắn cờ `city_temperature_iqr_outlier`), không phải lỗi đo lường.

**2. Xu hướng theo thời gian:**
- Nhiệt độ trung bình theo năm tăng dần với tốc độ **+0.0269°C/năm** trong giai đoạn 1863–2013, thể hiện xu hướng **nóng lên** nhất quán với biến đổi khí hậu toàn cầu (`reports/images/06_yearly_trend.png`).
- Tương quan tuyến tính của `year`/`decade` với target chỉ ở mức yếu (r ≈ 0.048): biên độ dao động theo mùa và theo vĩ độ trong cùng một năm lớn hơn nhiều biên độ thay đổi giữa các năm, nên xu hướng ấm lên có thật nhưng là tín hiệu nhỏ và dài hạn.
- Số quan sát mỗi năm tăng dần theo thời gian (năm 1863 chỉ có 28.650 dòng, các năm sau có hàng trăm nghìn dòng) do số thành phố được ghi nhận tăng dần theo thời gian, không phải do biến động khí hậu — cần lưu ý độ phủ dữ liệu (coverage bias) này khi diễn giải xu hướng dài hạn.

**3. Mùa vụ:**
- Tháng nóng nhất: **Tháng 7 (23.41°C)**; tháng lạnh nhất: **Tháng 1 (10.37°C)**; chênh lệch **13.04°C** (`reports/images/07_monthly_seasonality.png`).
- Độ lệch chuẩn theo tháng không đều: các tháng mùa đông (Th.12–Th.2) có std ≈ 11.9–12.9°C, trong khi mùa hè (Th.6–Th.8) chỉ ≈ 5.2–6.1°C — vào mùa đông, chênh lệch nhiệt độ giữa các thành phố (gần xích đạo vs vĩ độ cao) rất lớn, còn mùa hè các thành phố "hội tụ" gần nhau hơn.
- Tương quan tuyến tính của `month`/`quarter` với target chỉ ở mức yếu (r ≈ 0.10) dù biên độ mùa vụ (13.04°C) rất lớn, vì quan hệ giữa tháng và nhiệt độ là **phi tuyến/tuần hoàn** (tháng 12 và tháng 1 liền kề về khí hậu nhưng cách xa nhau về mặt số học) — hệ số tương quan Pearson không phản ánh đúng bản chất tuần hoàn này.

**4. Tương quan giữa các biến (multicollinearity):**
- Nhóm 4 biến `land_average_temperature`, `land_max_temperature`, `land_min_temperature`, `land_and_ocean_average_temperature` tương quan với nhau **r > 0.98** — gần như trùng lặp thông tin (đây là chỉ số nhiệt độ đất/đại dương toàn cầu theo tháng, không phân biệt theo thành phố).
- `major_city_average_temperature` tương quan **r = 1.000** với `city_average_temperature` — đây là **dấu hiệu rò rỉ (leakage)**: cột này chỉ có giá trị khi thành phố thuộc nhóm Major City (151.293/5.579.085 dòng, ~2,7%) và trùng khớp hoàn toàn với target khi có mặt. Không được dùng trực tiếp làm feature.
- `country_average_temperature` tương quan mạnh (r = 0.897) với target — hợp lý vì đây là trung bình nhiệt độ của quốc gia chứa thành phố đó; có thể dùng làm feature nhưng cần cân nhắc vì mức tương quan cao có thể gây dư thừa thông tin với các đặc trưng vị trí khác.
- `latitude` có tương quan âm vừa phải (r = -0.467) — mạnh nhất trong số các biến "thô" không phải nhiệt độ. Vì `latitude` ở đây có dấu (từ -52.24 đến 69.92, âm là Nam bán cầu), tương quan âm cho thấy thành phố càng lệch về Bắc bán cầu (vĩ độ dương lớn) thì nhiệt độ trung bình càng thấp — phù hợp vì tập 50 quốc gia có nhiều nước lạnh ở vĩ độ Bắc cao (Nga, các nước Bắc Âu) và nhiều nước nóng gần xích đạo ở vĩ độ thấp/âm (Ấn Độ, Brazil, Nigeria).

**5. Đặc điểm định danh (categorical):**
- `country_name`: đúng 50 quốc gia (theo thiết kế pipeline Notebook 02); dẫn đầu số dòng là India (693.711), China (686.966), United States (464.897), Brazil (392.342), Japan (316.032).
- `city_name`: 3.070 tên duy nhất nhưng **tên có thể trùng lặp giữa các quốc gia/vị trí khác nhau** (ví dụ Springfield, Worcester đều xuất hiện ở nhiều nơi khác nhau với số dòng cao bất thường) — xác nhận lại nhận định ở Notebook 01: `city_name` một mình không phải khóa định danh duy nhất, phải kết hợp với `country_name` + `latitude` + `longitude`.

### 4.2 Ý Nghĩa cho Dự Báo

- **Vĩ độ là driver địa lý mạnh nhất trong các biến thô**, nhưng vì `latitude` có dấu, bản chất khí hậu (càng xa xích đạo càng lạnh, bất kể Bắc hay Nam) có thể được biểu diễn tốt hơn bằng khoảng cách tuyệt đối tới xích đạo (`abs(latitude)`) thay vì giá trị có dấu.
- **Tháng/quý có quan hệ tuần hoàn, không tuyến tính** với nhiệt độ (biên độ mùa vụ 13.04°C nhưng r tuyến tính chỉ ~0.10) → không nên chỉ dùng `month` dạng số nguyên thô làm feature; cần mã hóa tuần hoàn (sin/cos) hoặc nhóm theo mùa để mô hình học đúng tính chu kỳ.
- **Xu hướng ấm lên là có thật nhưng nhỏ và dài hạn** (+0.0269°C/năm) — một đặc trưng biểu diễn thời gian trôi (ví dụ số năm kể từ mốc bắt đầu) có thể giúp mô hình tách tín hiệu này khỏi biến động mùa vụ/vị trí có biên độ lớn hơn nhiều.
- **Multicollinearity cao giữa các biến `land_*`** (r > 0.98) và **rủi ro rò rỉ từ `major_city_average_temperature`** (r = 1.000) là hai vấn đề cần xử lý ngay ở Notebook 05, trước khi đưa vào huấn luyện mô hình.
- **`city_name` không phải khóa định danh duy nhất** — nếu mã hóa vị trí theo tên thành phố, phải ghép thêm `country_name`/tọa độ để tránh gộp nhầm các thành phố trùng tên nhưng khác vị trí thực tế.
- **Độ bất định (uncertainty) giảm dần theo thời gian** (tương quan với year/decade từ -0.60 đến -0.88) phản ánh chất lượng đo lường được cải thiện qua các thời kỳ, gần như không liên quan đến giá trị nhiệt độ thực tế (r với target chỉ ≈ -0.09) — có thể cân nhắc dùng như đặc trưng chất lượng dữ liệu (data-quality feature) hơn là loại bỏ.

### 4.3 Các Kích Hoạt cho Feature Engineering

Những phát hiện trên sẽ được sử dụng ở Notebook 05 để:
1. **Tạo đặc trưng thời gian tuần hoàn:** mã hóa `month`/`quarter` bằng sin/cos hoặc nhóm `season`, thay vì dùng số nguyên thô, để phản ánh đúng biên độ mùa vụ 13.04°C.
2. **Tạo đặc trưng xu hướng dài hạn:** ví dụ `years_since_start` (số năm kể từ 1863) để mô hình tách được tín hiệu ấm lên +0.0269°C/năm khỏi biến động mùa vụ/vị trí.
3. **Tạo đặc trưng địa lý theo khoảng cách tới xích đạo:** `abs(latitude)` bên cạnh `latitude` có dấu, vì đây là driver tương quan mạnh nhất (r=-0.467) trong số biến thô.
4. **Loại bỏ nguy cơ rò rỉ và dư thừa:** không dùng trực tiếp `major_city_average_temperature` (r=1.000, rò rỉ); chỉ giữ lại **một** trong nhóm `land_average/max/min_temperature`, `land_and_ocean_average_temperature` (r>0.98 lẫn nhau) để tránh multicollinearity.
5. **Mã hóa vị trí đúng cách:** kết hợp `city_name` + `country_name` (+ tọa độ) khi tạo location-based encoding, vì `city_name` một mình có thể trùng giữa nhiều vị trí khác nhau.
6. **Cân nhắc dùng uncertainty như tín hiệu chất lượng dữ liệu**, không phải đặc trưng khí hậu, vì nó phản ánh giai đoạn đo lường (year/decade) hơn là bản thân nhiệt độ.

---

## V. Liên Kết Sang Notebook 05: Feature Engineering

### 5.1 Tóm Tắt

Notebook 04 đã hoàn thành EDA và phát hiện:
- ✅ Các đặc điểm cơ bản của dữ liệu
- ✅ Mối quan hệ giữa các biến
- ✅ Xu hướng theo thời gian
- ✅ Mùa vụ và tính chất thời gian

### 5.2 Chuẩn Bị cho Notebook 05

**Notebook 05 (Feature Engineering) sẽ:**
1. Sử dụng các phát hiện từ EDA để tạo đặc trưng mới
2. Loại bỏ đặc trưng dư thừa (multicollinearity)
3. Tạo polynomial features, lag features, rolling averages
4. Chuẩn bị dữ liệu cho Notebook 06 (Machine Learning)

**Các đặc trưng dự kiến:**
- Time-based: `month`, `quarter`, `season`, `is_summer`, `is_winter`
- Trend: `year_normalized`, polynomial terms, moving averages
- Lag: previous month/year temperature
- Statistical: rolling mean, rolling std

### 5.3 Quy Trình Tiếp Theo

```
04_eda_visualization (Hiện tại) ✓
         ↓
05_feature_engineering (Tiếp theo)
         ↓
06_machine_learning (Train mô hình)
         ↓
07_prediction_demo (Thực hiện dự báo)
```

---

## VI. Ghi Chú và Hướng Dẫn Chạy Notebook

### Cách Chạy Notebook

1. **Đảm bảo Notebook 03 đã chạy:** để tạo file `cleaned_city_temperature.csv` hoặc table PostgreSQL
2. **Chạy từ trên xuống (Run All):** Ctrl+Shift+P → "Run All Cells"
3. **Kiểm tra output:** Xem messages để xác nhận dữ liệu đã được tải
4. **Kiểm tra ảnh:** Các biểu đồ sẽ được lưu trong `reports/images/`

### Xử Lý Lỗi Thường Gặp

| Lỗi | Nguyên Nhân | Giải Pháp |
|---|---|---|
| `ModuleNotFoundError: No module named 'psycopg2'` | Thiếu PostgreSQL driver | `pip install psycopg2-binary python-dotenv` |
| `FileNotFoundError: cleaned_city_temperature.csv` | Notebook 03 chưa chạy | Chạy Notebook 03 trước |
| `ConnectionError` từ PostgreSQL | Database không chạy | Kiểm tra PostgreSQL, hoặc notebook sẽ fallback sang CSV |
| Biểu đồ không hiển thị | Jupyter kernel hang | Restart kernel |
| Target không được tìm thấy | Dữ liệu không có cột city_average_temperature | Kiểm tra tên cột trong dữ liệu, hoặc chỉnh sửa logic ưu tiên trong cell 18 |

### Tùy Chỉnh

- **Thay đổi biến mục tiêu:** Sửa logic ưu tiên trong cell 18 (tìm dòng `if 'city_average_temperature' in df.columns:`)
- **Thay đổi số lượng bins:** Sửa công thức `n_bins = max(20, min(100, n_data // 100 + 30))` 
- **Lưu thêm ảnh:** Thêm `plt.savefig(REPORT_IMAGES / 'tên_file.png')` trước `plt.show()`
- **Tối ưu kết nối PostgreSQL:** Timeout mặc định là 5 giây, có thể sửa trong cell 8 (`connect_timeout=5`)